# 1. Preparation

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
np.random.seed(42)
print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
rng = np.random.default_rng(42)
n_users = 10000
df = pd.DataFrame({
    "user_id": np.arange(1, n_users + 1),
    "group": rng.permutation(["A"] * 5000 + ["B"] * 5000)
})

parameters = {
    "A": {
        "ctr": 0.20,
        "registration": 0.40,
        "d1_retention": 0.55,
        "d7_retention": 0.30},
    "B": {
        "ctr": 0.24,
        "registration": 0.45,
        "d1_retention": 0.60,
        "d7_retention": 0.35}}

df["exposed"] = 1
df["clicked"] = 0
df["registered"] = 0
df["retained_d1"] = 0
df["retained_d7"] = 0

for group, p in parameters.items():
    mask = df["group"] == group
    n = mask.sum()

    clicks = rng.binomial(1, p["ctr"], n)
    df.loc[mask, "clicked"] = clicks

    click_mask = mask & (df["clicked"] == 1)
    n_clicked = click_mask.sum()

    registrations = rng.binomial(1, p["registration"], n_clicked)
    df.loc[click_mask, "registered"] = registrations

    registered_mask = mask & (df["registered"] == 1)
    n_registered = registered_mask.sum()

    df.loc[registered_mask, "retained_d1"] = (rng.binomial(1, p["d1_retention"], n_registered))

    df.loc[registered_mask, "retained_d7"] = (rng.binomial(1, p["d7_retention"], n_registered))
print("Dataset shape:", df.shape)
print("\nGroup distribution:")
print(df["group"].value_counts())
print("\nFirst 10 observations:")
display(df.head(10))

Dataset shape: (10000, 7)

Group distribution:
group
B    5000
A    5000
Name: count, dtype: int64

First 10 observations:


,user_id,group,exposed,clicked,registered,retained_d1,retained_d7
0,1,B,1,1,1,1,0
1,2,B,1,0,0,0,0
2,3,A,1,0,0,0,0
3,4,B,1,0,0,0,0
4,5,A,1,0,0,0,0
5,6,B,1,0,0,0,0
6,7,B,1,0,0,0,0
7,8,B,1,0,0,0,0
8,9,A,1,1,1,0,0
9,10,A,1,1,0,0,0


# 2. Data Cleaning & Validation

In [3]:
# 1. Check missing values
print("Missing values:")
print(df.isnull().sum())

# 2. Check duplicate users
print("\nDuplicate user IDs:")
print(df["user_id"].duplicated().sum())

# 3. Check group distribution
print("\nGroup distribution:")
print(df["group"].value_counts())

# 4. Validate user behavior consistency
invalid_registration = (
    (df["registered"] == 1) &
    (df["clicked"] == 0)
).sum()

invalid_d1 = (
    (df["retained_d1"] == 1) &
    (df["registered"] == 0)
).sum()

invalid_d7 = (
    (df["retained_d7"] == 1) &
    (df["registered"] == 0)
).sum()

print("\nBehavior consistency checks:")
print("Registered without clicking:", invalid_registration)
print("D1 retained without registration:", invalid_d1)
print("D7 retained without registration:", invalid_d7)

# 5. Check data types
print("\nData types:")
print(df.dtypes)

# 6. Final validation
assert len(df) == 10000
assert df["user_id"].is_unique
assert df["group"].value_counts().to_dict() == {"A": 5000, "B": 5000}
assert df.isnull().sum().sum() == 0
assert invalid_registration == 0
assert invalid_d1 == 0
assert invalid_d7 == 0

print("\nAll data quality checks passed!")

Missing values:
user_id        0
group          0
exposed        0
clicked        0
registered     0
retained_d1    0
retained_d7    0
dtype: int64

Duplicate user IDs:
0

Group distribution:
group
B    5000
A    5000
Name: count, dtype: int64

Behavior consistency checks:
Registered without clicking: 0
D1 retained without registration: 0
D7 retained without registration: 0

Data types:
user_id         int64
group          object
exposed         int64
clicked         int64
registered      int64
retained_d1     int64
retained_d7     int64
dtype: object

All data quality checks passed!


# 3. Conversion Funnel Analysis

In [4]:
# Aggregate user behavior by experiment group
funnel = df.groupby("group").agg(
    Exposures=("exposed", "sum"),
    Clicks=("clicked", "sum"),
    Registrations=("registered", "sum")
)

# Calculate conversion metrics
funnel["CTR"] = (
    funnel["Clicks"] / funnel["Exposures"]
)

funnel["Click_to_Registration"] = (
    funnel["Registrations"] / funnel["Clicks"]
)

funnel["Overall_Conversion"] = (
    funnel["Registrations"] / funnel["Exposures"]
)

print("Conversion Funnel Analysis:")

display(
    funnel.style.format({
        "CTR": "{:.2%}",
        "Click_to_Registration": "{:.2%}",
        "Overall_Conversion": "{:.2%}"
    })
)

# Calculate treatment effect (B vs A)
metrics = [
    "CTR",
    "Click_to_Registration",
    "Overall_Conversion"
]

comparison = pd.DataFrame({
    "Group_A": funnel.loc["A", metrics],
    "Group_B": funnel.loc["B", metrics]
})

comparison["Absolute_Lift_pp"] = (
    comparison["Group_B"] - comparison["Group_A"]
) * 100

comparison["Relative_Lift_%"] = (
    (comparison["Group_B"] / comparison["Group_A"]) - 1
) * 100

print("\nTreatment Effect Comparison:")
display(comparison.round(2))

Conversion Funnel Analysis:


,Exposures,Clicks,Registrations,CTR,Click_to_Registration,Overall_Conversion
group,,,,,,
A,5000,985,377,19.70%,38.27%,7.54%
B,5000,1240,595,24.80%,47.98%,11.90%



Treatment Effect Comparison:


,Group_A,Group_B,Absolute_Lift_pp,Relative_Lift_%
CTR,0.20,0.25,5.10,25.89
Click_to_Registration,0.38,0.48,9.71,25.37
Overall_Conversion,0.08,0.12,4.36,57.82


# 4. Statistical Hypothesis Testing

In [5]:
# Primary Metric: Overall Registration Conversion
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import norm
import numpy as np

# Extract conversion counts and sample sizes
success_A = int(funnel.loc["A", "Registrations"])
success_B = int(funnel.loc["B", "Registrations"])

n_A = int(funnel.loc["A", "Exposures"])
n_B = int(funnel.loc["B", "Exposures"])

p_A = success_A / n_A
p_B = success_B / n_B

# Two-proportion Z-test (two-sided)
z_stat, p_value = proportions_ztest(
    count=[success_B, success_A],
    nobs=[n_B, n_A],
    alternative="two-sided")

# Calculate 95% confidence interval
alpha = 0.05
z_critical = norm.ppf(1 - alpha / 2)
difference = p_B - p_A
standard_error = np.sqrt(p_A * (1 - p_A) / n_A +p_B * (1 - p_B) / n_B)
ci_lower = difference - z_critical * standard_error
ci_upper = difference + z_critical * standard_error

print("A/B Testing: Overall Registration Conversion")
print("-" * 48)
print(f"Group A conversion rate: {p_A:.2%}")
print(f"Group B conversion rate: {p_B:.2%}")
print(f"\nAbsolute lift: {difference * 100:.2f} pp")
print(f"Relative lift: {(p_B / p_A - 1) * 100:.2f}%")
print(f"\nZ-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.6g}")
print(f"95% CI for absolute lift: "f"[{ci_lower * 100:.2f}, {ci_upper * 100:.2f}] pp")

# Statistical conclusion
if p_value < alpha:
    print("\nDecision: Reject H0")
    print("The difference is statistically significant.")
else:
    print("\nDecision: Fail to reject H0")
    print("Insufficient evidence of a difference.")

A/B Testing: Overall Registration Conversion
------------------------------------------------
Group A conversion rate: 7.54%
Group B conversion rate: 11.90%

Absolute lift: 4.36 pp
Relative lift: 57.82%

Z-statistic: 7.3591
P-value: 1.85087e-13
95% CI for absolute lift: [3.20, 5.52] pp

Decision: Reject H0
The difference is statistically significant.


# 5. User Retention Analysis

In [6]:
from statsmodels.stats.proportion import proportions_ztest

# Analyze retention among registered users
registered_users = df[df["registered"] == 1].copy()
retention = registered_users.groupby("group").agg(
    Registered_Users=("user_id", "count"),
    D1_Retained=("retained_d1", "sum"),
    D7_Retained=("retained_d7", "sum"))

# Calculate retention rates
retention["D1_Retention_Rate"] = (retention["D1_Retained"] /retention["Registered_Users"])

retention["D7_Retention_Rate"] = (retention["D7_Retained"] / retention["Registered_Users"])

print("User Retention Analysis:")

display(
    retention.style.format({
        "D1_Retention_Rate": "{:.2%}",
        "D7_Retention_Rate": "{:.2%}"
    }))

# Compare retention rates
retention_comparison = pd.DataFrame({
    "Group_A": retention.loc[
        "A", ["D1_Retention_Rate", "D7_Retention_Rate"]
    ],
    "Group_B": retention.loc[
        "B", ["D1_Retention_Rate", "D7_Retention_Rate"]
    ]
})

retention_comparison["Absolute_Lift_pp"] = (
    retention_comparison["Group_B"] -
    retention_comparison["Group_A"]
) * 100

retention_comparison["Relative_Lift_%"] = (
    retention_comparison["Group_B"] /
    retention_comparison["Group_A"] - 1
) * 100

print("\nRetention Comparison:")
display(retention_comparison.round(2))

# Exploratory significance tests
print("\nExploratory Retention Significance Tests:")

for day in ["D1", "D7"]:
    count = [
        retention.loc["B", f"{day}_Retained"],
        retention.loc["A", f"{day}_Retained"]
    ]

    nobs = [
        retention.loc["B", "Registered_Users"],
        retention.loc["A", "Registered_Users"]
    ]

    z_stat, p_value = proportions_ztest(
        count=count,
        nobs=nobs,
        alternative="two-sided"
    )

    print(f"\n{day} Retention:")
    print(f"Z-statistic: {z_stat:.4f}")
    print(f"P-value: {p_value:.6g}")

    if p_value < 0.05:
        print("Statistically significant difference.")
    else:
        print("No statistically significant difference detected.")

User Retention Analysis:


,Registered_Users,D1_Retained,D7_Retained,D1_Retention_Rate,D7_Retention_Rate
group,,,,,
A,377,210,121,55.70%,32.10%
B,595,338,213,56.81%,35.80%



Retention Comparison:


,Group_A,Group_B,Absolute_Lift_pp,Relative_Lift_%
D1_Retention_Rate,0.56,0.57,1.1,1.98
D7_Retention_Rate,0.32,0.36,3.7,11.54



Exploratory Retention Significance Tests:

D1 Retention:
Z-statistic: 0.3381
P-value: 0.735267
No statistically significant difference detected.

D7 Retention:
Z-statistic: 1.1844
P-value: 0.23624
No statistically significant difference detected.


# 6. Data Visualization


6.1 Conversion Funnel Visualization

In [7]:
import plotly.graph_objects as go

fig1 = go.Figure()
colors = {
    "A": "#64748B",
    "B": "#2563EB"
}

for group in ["A", "B"]:
    fig1.add_trace(
        go.Funnel(
            name=f"Group {group}",
            y=["Exposure", "Click", "Registration"],
            x=[
                funnel.loc[group, "Exposures"],
                funnel.loc[group, "Clicks"],
                funnel.loc[group, "Registrations"]
            ],
            textinfo="value+percent initial",
            marker=dict(color=colors[group])
        )
    )

fig1.update_layout(
    title="A/B Testing: User Conversion Funnel",
    font=dict(size=14),
    template="plotly_white",
    width=1000,
    height=600,
    funnelmode="group",
    legend_title="Experiment Group"
)

fig1.show()

6.2 Conversion Rate Comparison

In [8]:
import plotly.graph_objects as go
metrics = [
    "CTR",
    "Click_to_Registration",
    "Overall_Conversion"
]

labels = [
    "Click-Through Rate",
    "Click-to-Registration",
    "Overall Conversion"
]

rates_A = [funnel.loc["A", m] * 100 for m in metrics]
rates_B = [funnel.loc["B", m] * 100 for m in metrics]

fig2 = go.Figure()

fig2.add_trace(go.Bar(
    name="Group A (Control)",
    x=labels,
    y=rates_A,
    marker_color="#64748B",
    text=[f"{v:.2f}%" for v in rates_A],
    textposition="outside"
))

fig2.add_trace(go.Bar(
    name="Group B (Treatment)",
    x=labels,
    y=rates_B,
    marker_color="#2563EB",
    text=[f"{v:.2f}%" for v in rates_B],
    textposition="outside"
))

fig2.update_layout(
    title="A/B Testing: Conversion Rate Comparison",
    xaxis_title="Conversion Metrics",
    yaxis_title="Conversion Rate (%)",
    yaxis=dict(range=[0, 60]),
    barmode="group",
    template="plotly_white",
    width=1000,
    height=600,
    font=dict(size=14),
    legend_title="Experiment Group"
)

fig2.show()

6.3 User Retention Comparison


In [9]:
import plotly.graph_objects as go
metrics = ["D1_Retention_Rate", "D7_Retention_Rate"]
labels = ["Day 1 Retention", "Day 7 Retention"]

rates_A = [retention.loc["A", m] * 100 for m in metrics]
rates_B = [retention.loc["B", m] * 100 for m in metrics]

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    name="Group A (Control)",
    x=labels,
    y=rates_A,
    marker_color="#64748B",
    text=[f"{v:.2f}%" for v in rates_A],
    textposition="outside"
))

fig3.add_trace(go.Bar(
    name="Group B (Treatment)",
    x=labels,
    y=rates_B,
    marker_color="#2563EB",
    text=[f"{v:.2f}%" for v in rates_B],
    textposition="outside"
))

fig3.update_layout(
    title="A/B Testing: User Retention Comparison",
    xaxis_title="Retention Metrics",
    yaxis_title="Retention Rate (%)",
    yaxis=dict(range=[0, 70]),
    barmode="group",
    template="plotly_white",
    width=1000,
    height=600,
    font=dict(size=14),
    legend_title="Experiment Group",
    annotations=[
        dict(
            x="Day 1 Retention",
            y=63,
            text="p = 0.7353 (Not Significant)",
            showarrow=False,
            font=dict(size=12)
        ),
        dict(
            x="Day 7 Retention",
            y=43,
            text="p = 0.2362 (Not Significant)",
            showarrow=False,
            font=dict(size=12)
        )
    ]
)

fig3.show()

# 7. Export Interactive Visualizations


In [10]:
from pathlib import Path
from google.colab import files

# Create output directory
output_dir = Path("visualizations")
output_dir.mkdir(exist_ok=True)

# Export interactive HTML charts
charts = {
    "conversion_funnel": fig1,
    "conversion_rate_comparison": fig2,
    "retention_comparison": fig3}
for name, fig in charts.items():
    file_path = output_dir / f"{name}.html"
    fig.write_html(
        str(file_path),
        include_plotlyjs=True,
        full_html=True)
    print(f"Exported: {file_path}")
print("\nAll interactive charts exported successfully!")


Exported: visualizations/conversion_funnel.html
Exported: visualizations/conversion_rate_comparison.html
Exported: visualizations/retention_comparison.html

All interactive charts exported successfully!
